# bibr 🦫 — Python API Client Demo

This notebook demonstrates how to call the bibr REST API from Python.

**Prerequisites:**
- bibr API server running: `uv run bibr serve` (or via Docker)
- `httpx` installed: `uv pip install httpx`

By default this connects to `http://localhost:8000`. Change `BASE_URL` below if
your server is running elsewhere.

## Configuration

In [ ]:
import os

import httpx

BASE_URL = os.getenv("BIBR_API_URL", "http://localhost:8000")
API_KEY = os.getenv("BIBR_API_KEY", "")  # Match the server's AUTH_API_KEY
FILE_PATH = os.getenv("PAPER_PATH", "paper.pdf")

headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

## 1. Health check

Verify the server is running and ready.

In [ ]:
r = httpx.get(f"{BASE_URL}/health", headers=headers)
print(f"Health: {r.status_code}")

r = httpx.get(f"{BASE_URL}/ready", headers=headers)
print(f"Ready:  {r.json()}")

## 2. Extract metadata (JSON)

`POST /papers/extract` accepts a multipart upload and returns the bibr v11
JSON schema synchronously. Optional form fields `start_page`, `end_page`,
and `figure_images` (default `false`) tune the extraction.

In [ ]:
import os

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"File not found: {FILE_PATH}")

print(f"Sending {os.path.basename(FILE_PATH)} to {BASE_URL}/papers/extract ...")

with open(FILE_PATH, "rb") as f:
    response = httpx.post(
        f"{BASE_URL}/papers/extract",
        headers=headers,
        files={"file": (os.path.basename(FILE_PATH), f, "application/pdf")},
        timeout=300,
    )
response.raise_for_status()
data = response.json()
print("Extraction complete!")

In [ ]:
metadata = data["metadata"]
print(f"Title:      {metadata['title']}")
print(f"DOI:        {metadata['doi']}")
print(f"Keywords:   {metadata['keywords']}")
print(f"Authors:    {len(data['author'])}")
print(f"References: {len(data['bib'])}")
print(f"Sections:   {len(data['section'])}")

In [ ]:
# Authors
for a in data["author"]:
    print(f"  {a['given']} {a['family']} — {a.get('affiliation', '')}")

In [ ]:
# Sections with IMRaD classification
for s in data["section"]:
    score = s.get("classification_score", 0)
    print(f"  [{s.get('section_type', 'unknown'):>15}] {s['header']}  ({score:.0%})")

In [ ]:
# References (first 10)
for ref in data["bib"][:10]:
    print(f"  [{ref['bib_id']}] {ref['authors']} ({ref['year']}). {ref['title']}")

if len(data["bib"]) > 10:
    print(f"  ... and {len(data['bib']) - 10} more")

## 3. Save extraction result to disk

In [ ]:
import json
import tempfile

output_path = os.path.join(tempfile.gettempdir(), "bibr_api_export.json")

with open(output_path, "w") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Saved JSON to {output_path}")

## 4. Processing without a server

For direct processing without running an API server, use the `bibr` CLI:

```bash
# Install from the source checkout and configure your hardware
uv sync --extra all
uv run bibr setup

# Process a file directly
uv run bibr chew paper.pdf -o result.json
```

The setup wizard chooses the OCR and LLM runtimes for your hardware.
See the [installation guide](../docs/getting-started/install.md).